# Pandas: From Basics to Advanced Data Wrangling

A comprehensive guide covering everything from loading data to advanced transformations.
This single notebook replaces an entire series — all the Pandas you need in one place.


## 1. Setup & Data Loading


In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

print('Pandas', pd.__version__, '| NumPy', np.__version__)


### CSV, Excel, JSON — The Three Musketeers


In [ ]:
# CSV: the universal format
csv_data = """date,product,category,revenue,units
2024-01-01,Widget,Electronics,1200,15
2024-01-02,Gadget,Clothing,850,10
2024-01-03,Doohickey,Food,2000,25
2024-01-04,Widget,Electronics,950,12
2024-01-05,Whatsit,Books,300,5"""

df = pd.read_csv(StringIO(csv_data))
df


### Key Parameters for Real Files


In [ ]:
params = """
pd.read_csv('file.csv', sep=',', header=0, index_col=None,
             na_values=['NA', 'Missing'], parse_dates=['date'],
             dtype={'units': int}, nrows=1000)

pd.read_excel('file.xlsx', sheet_name=['Sheet1', 'Sheet2'])
pd.read_json('file.json')
pd.read_sql('SELECT * FROM table', connection)
"""
print(params)


## 2. Exploring Your Data


In [ ]:
print('Shape:', df.shape)
print('\nColumn types:'); print(df.dtypes)
print('\nFirst 5:'); print(df.head())
print('\nSummary stats:'); print(df.describe(include='all'))
print('\nMissing values:'); print(df.isnull().sum())


## 3. Indexing & Selection


In [ ]:
# Label-based (.loc) vs Position-based (.iloc)
print('loc by label:', '\n', df.loc[0:2, ['product', 'revenue']])
print('\niloc by position:', '\n', df.iloc[0:3, 0:3])
print('\nBoolean filter:', '\n', df[df['revenue'] > 1000])


In [ ]:
# Set and use a datetime index
df_idx = df.copy()
df_idx['date'] = pd.to_datetime(df_idx['date'])
df_idx = df_idx.set_index('date')
print('After DatetimeIndex:')
print(df_idx.loc['2024-01-01':'2024-01-03'])


## 4. Handling Missing Data


In [ ]:
messy = df.copy()
messy.loc[1, 'revenue'] = np.nan
messy.loc[3, 'units'] = np.nan
print('Before cleaning:'); print(messy)
print('\nAfter fillna(0):'); print(messy.fillna(0))
print('\nAfter dropna:'); print(messy.dropna())
print('\nForward fill:'); print(messy.ffill())


## 5. Text Operations


In [ ]:
print(df['product'].str.upper().tolist())
print(df['category'].str.contains('Elec'))
print(df['product'].str.extract(r'(\w+)'))


## 6. Merge, Join & Concatenate


In [ ]:
left = pd.DataFrame({'id': [1,2,3], 'name': ['A','B','C']})
right = pd.DataFrame({'id': [2,3,4], 'score': [85,92,78]})

print('Inner merge:'); print(pd.merge(left, right, on='id', how='inner'))
print('\nLeft merge:'); print(pd.merge(left, right, on='id', how='left'))
print('\nOuter merge:'); print(pd.merge(left, right, on='id', how='outer'))


In [ ]:
q1 = pd.DataFrame({'sales':[100,150]}, index=['A','B'])
q2 = pd.DataFrame({'sales':[120,140]}, index=['A','B'])
print('Concatenated:'); print(pd.concat([q1, q2], axis=1, keys=['Q1','Q2']))


## 7. Reshaping: Pivot & Melt


In [ ]:
sales = pd.DataFrame({
    'product': ['A','A','B','B'],
    'quarter': ['Q1','Q2','Q1','Q2'],
    'revenue': [100, 120, 150, 160]
})
pivoted = sales.pivot(index='product', columns='quarter', values='revenue')
print('Pivoted (long -> wide):'); print(pivoted)
print('\nMelted back (wide -> long):'); print(pivoted.reset_index().melt(id_vars='product', var_name='quarter', value_name='revenue'))


## 8. GroupBy: Split-Apply-Combine


In [ ]:
print(df.groupby('category')['revenue'].agg(['sum', 'mean', 'count']))
print('\n---')
print(df.groupby('category').agg({'revenue': 'sum', 'units': 'mean'}))


In [ ]:
# Transform — group-aware, same shape
df['revenue_pct'] = df.groupby('category')['revenue'].transform(lambda x: x / x.sum())
df


## 9. Time Series: Resampling & Rolling


In [ ]:
dates = pd.date_range('2024-01-01', periods=30, freq='D')
ts = pd.DataFrame({'value': np.random.randint(50, 150, 30)}, index=dates)

print('Weekly sum:'); print(ts.resample('W').sum().head())
print('\nRolling 7-day mean:'); print(ts['value'].rolling(7).mean().head(10))


## 10. Apply & Map — Custom Transformations


In [ ]:
# map() for Series transformations
df['product_code'] = df['product'].map({'Widget': 'W', 'Gadget': 'G', 'Doohickey': 'D', 'Whatsit': 'T'})

# Custom row-level logic with apply()
def classify(row):
    if row['revenue'] > 1500: return 'High'
    if row['revenue'] > 1000: return 'Medium'
    return 'Low'
df['class'] = df.apply(classify, axis=1)
df


## 11. Styling Pandas Tables

Visual formatting for DataFrames in Jupyter — useful for reports and presentations.


In [ ]:
# Sample data for styling
styling_df = pd.DataFrame({
    'product': ['A', 'B', 'C', 'D', 'E'],
    'q1': [120, 90, 200, 150, 80],
    'q2': [130, 85, 210, 160, 95],
    'q3': [125, 95, 205, 155, 85],
    'q4': [140, 100, 220, 170, 90]
})
styling_df


### Highlight Min/Max


In [ ]:
styling_df.style.highlight_max(color='lightgreen', axis=0).highlight_min(color='salmon', axis=0)


### Color Gradients (Heatmap)


In [ ]:
styling_df.style.background_gradient(cmap='Blues', subset=['q1', 'q2', 'q3', 'q4'])


### Bar Charts Inside Cells


In [ ]:
styling_df.style.bar(subset=['q1', 'q2', 'q3', 'q4'], color='#5fba7d')


### Custom Conditional Formatting


In [ ]:
def color_high(val):
    if val > 150:
        return 'background-color: gold; font-weight: bold'
    return ''

styling_df.style.applymap(color_high, subset=['q1', 'q2', 'q3', 'q4'])


### Number Formatting


In [ ]:
styling_df.style.format({'q1': '${:,.0f}', 'q2': '${:,.0f}', 'q3': '${:,.0f}', 'q4': '${:,.0f}'})


### Chaining Multiple Styles


In [ ]:
(styling_df.style
 .background_gradient(cmap='Greens', subset=['q1', 'q2'])
 .background_gradient(cmap='Oranges', subset=['q3', 'q4'])
 .highlight_max(color='yellow', axis=None)
 .format('${:.0f}', subset=['q1', 'q2', 'q3', 'q4'])
 .set_caption('Quarterly Sales Report'))


## Summary: When to Use What


In [ ]:
print('''
Operation              | Method                  | Best For
-----------------------|-------------------------|--------------------------
Filter rows            | df[condition] / .query()| Simple or complex filters
Select columns         | df['col'] / df[cols]   | Column subsetting
Label-based access     | .loc[row, col]         | Named index access
Position access        | .iloc[row, col]        | Integer positions
Missing data           | .fillna() / .dropna()  | Clean incomplete records
Merge tables           | pd.merge()             | SQL-style joins
Stack rows             | pd.concat(axis=0)      | Append datasets
Reshape                | .pivot() / .melt()     | Change layout
Group + aggregate      | .groupby().agg()       | Summary statistics
Element-wise transform | .map() / .apply()      | Custom logic
Time series window     | .rolling() / .resample()| Trend analysis
Style tables           | .style.background_gradient() | Visual formatting
''')
